# Entrenamiento RF-DETR en Colab (Nano · Small)

Entrena **RF-DETR Nano** y **RF-DETR Small** sobre el mismo dataset de tarjetas.

- Dataset: **Cards** (mismo que YOLO11 en el otro notebook)
- Formato de dataset: **YOLO nativo** — no se requiere conversión
- Split reproducible `train / val / test`
- Análisis de distribución de clases antes de entrenar
- Preview visual de augmentaciones antes de aplicarlas
- Export `.pt` y `.onnx` con guardado automático en Drive

> **Nota**: RF-DETR usa arquitectura basada en transformers (sin anclas).<br>
> Comparado con YOLO, suele ser más preciso en objetos superpuestos<br>
> pero requiere más VRAM. La T4 de Colab free admite Nano y Small con `batch_size=4`.

## 1) Instalación

In [ ]:
%pip install -q "rfdetr[train,loggers]" albumentations opencv-python pyyaml onnx roboflow
# protobuf>=4.25 es requerido por pytorch-lightning
%pip install -q "protobuf>=4.25,<6" --upgrade

import os, shutil, random, yaml, glob, json, math, colorsys
from pathlib import Path
from datetime import datetime
import importlib.metadata

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from IPython.display import Image as IPImage, display
from google.colab import drive

drive.mount('/content/drive')

# Verificar version instalada de rfdetr
try:
    rfdetr_version = importlib.metadata.version('rfdetr')
except importlib.metadata.PackageNotFoundError:
    rfdetr_version = 'desconocida'
print('rfdetr version:', rfdetr_version)

from rfdetr import RFDETRNano, RFDETRSmall
print('Modelos disponibles: RFDETRNano, RFDETRSmall')

## 1b) Descargar dataset desde Roboflow

Descarga el dataset directamente desde Roboflow para garantizar compatibilidad con los paquetes instalados en este entorno.

> Configura el secreto `ROBOFLOW_API_KEY` en **Colab → Secrets** antes de ejecutar.

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)

# Reemplaza con tus datos reales de proyecto
project = rf.workspace("thesis-s4jik").project("cards-4ceff")
version = project.version(3)
dataset = version.download("yolo26")
DATASET = dataset.location

print("Dataset descargado con éxito.")
print("DATASET:", DATASET)

## 2) Configuración

In [ ]:
import torch

# ── Rutas ──────────────────────────────────────────────────────────────────
# DATASET se define en la sección 1b (descarga de Roboflow)
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Cards_Training_Output'

BASE_DIR  = Path('/content/cards_rfdetr')
SPLIT_DIR = BASE_DIR / 'rfdetr_split'
AUG_DIR   = BASE_DIR / 'rfdetr_split_aug'
RUNS_DIR  = BASE_DIR / 'runs'

# ── Split ──────────────────────────────────────────────────────────────────
VAL_RATIO  = 0.15
TEST_RATIO = 0.10
SEED       = 42

# ── Reproducibilidad ───────────────────────────────────────────────────────
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── RF-DETR training ───────────────────────────────────────────────────────
# Cada checkpoint tiene sus propios positional embeddings a una resolución fija.
# Usar otra resolución causa RuntimeError en load_state_dict (size mismatch).
# Ambos valores son múltiplos de 32 (patch_size=16 * num_windows=2).
#
#   rf-detr-nano.pth  → [1,  577, 384] = 24×24 + 1 cls  →  384px
#   rf-detr-small.pth → [1, 1025, 384] = 32×32 + 1 cls  →  512px
IMG_SIZE_NANO  = 384   # 384 / 32 = 12  ✓
IMG_SIZE_SMALL = 512   # 512 / 32 = 16  ✓

EPOCHS     = 50     # RF-DETR converge mas rapido que YOLO
BATCH      = 4      # T4 Colab free: batch=4 es seguro para Nano y Small
GRAD_ACCUM = 4      # batch efectivo = BATCH * GRAD_ACCUM = 16
LR         = 1e-4

# ── Albumentaciones offline ────────────────────────────────────────────────
USE_OFFLINE_ALBUMENTATIONS = True
AUG_MULTIPLIER = 1

BASE_DIR.mkdir(parents=True, exist_ok=True)
print('Configuracion aplicada')
print('IMG_SIZE Nano: {}px  |  IMG_SIZE Small: {}px'.format(IMG_SIZE_NANO, IMG_SIZE_SMALL))
print('Batch efectivo: {} (batch={} x grad_accum={})'.format(BATCH * GRAD_ACCUM, BATCH, GRAD_ACCUM))

## 3) Utilidades

In [ ]:
# === Labels YOLO ============================================================

def read_yolo_labels(label_path):
    bboxes, class_ids = [], []
    if not Path(label_path).exists():
        return bboxes, class_ids
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, x, y, w, h = parts
            class_ids.append(int(cls))
            bboxes.append([float(x), float(y), float(w), float(h)])
    return bboxes, class_ids

def write_yolo_labels(label_path, bboxes, class_ids):
    with open(label_path, 'w') as f:
        for cls, box in zip(class_ids, bboxes):
            x, y, w, h = box
            f.write('{} {:.6f} {:.6f} {:.6f} {:.6f}\n'.format(cls, x, y, w, h))

# === Estructura de directorios ==============================================

def ensure_split_structure(root):
    for s in ['train', 'val', 'test']:
        (root / s / 'images').mkdir(parents=True, exist_ok=True)
        (root / s / 'labels').mkdir(parents=True, exist_ok=True)

def load_names_from_yaml(dataset_yaml):
    with open(dataset_yaml) as f:
        return yaml.safe_load(f)['names']

# === Split ==================================================================

def split_single_dataset(dataset_path, output_dir, val_ratio=0.15, test_ratio=0.10, seed=42):
    ensure_split_structure(output_dir)
    pairs = []
    for img in sorted(glob.glob(str(dataset_path) + '/train/images/*')):
        lbl = str(dataset_path) + '/train/labels/' + Path(img).stem + '.txt'
        if Path(lbl).exists():
            pairs.append((img, lbl))
    rnd = random.Random(seed)
    rnd.shuffle(pairs)
    n       = len(pairs)
    n_test  = int(n * test_ratio)
    n_val   = int(n * val_ratio)
    n_train = n - n_val - n_test
    splits  = {
        'train': pairs[:n_train],
        'val':   pairs[n_train:n_train + n_val],
        'test':  pairs[n_train + n_val:]
    }
    ds_name = Path(dataset_path).name
    for sname, items in splits.items():
        for i, (img, lbl) in enumerate(items):
            ext  = Path(img).suffix.lower()
            stem = ds_name + '_' + Path(img).stem + '_' + str(i)
            shutil.copy2(img, output_dir / sname / 'images' / (stem + ext))
            shutil.copy2(lbl, output_dir / sname / 'labels' / (stem + '.txt'))
    return {k: len(v) for k, v in splits.items()}

# === Augmentacion offline ===================================================

def apply_offline_augment(split_dir, aug_dir, aug_multiplier, transform):
    if aug_dir.exists():
        shutil.rmtree(aug_dir)
    shutil.copytree(split_dir, aug_dir)
    img_dir = aug_dir / 'train' / 'images'
    lbl_dir = aug_dir / 'train' / 'labels'
    created = 0
    for img_path in sorted(img_dir.glob('*')):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        bboxes, cls_ids = read_yolo_labels(lbl_path)
        if not bboxes:
            continue
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        for k in range(aug_multiplier):
            aug = transform(image=image, bboxes=bboxes, class_labels=cls_ids)
            out = cv2.cvtColor(aug['image'], cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(img_dir / (img_path.stem + '_aug' + str(k) + '.jpg')), out)
            write_yolo_labels(
                lbl_dir / (img_path.stem + '_aug' + str(k) + '.txt'),
                aug['bboxes'], [int(c) for c in aug['class_labels']]
            )
            created += 1
    return created

def build_yaml(data_root, names, yaml_path):
    cfg = {
        'path':  str(data_root),
        'train': 'train/images',
        'val':   'val/images',
        'test':  'test/images',
        'nc':    len(names),
        'names': names
    }
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

# === Colores ================================================================

def _class_colors(n):
    return [
        tuple(int(c * 255) for c in colorsys.hsv_to_rgb(i / max(n, 1), 0.75, 0.9))
        for i in range(n)
    ]

# === Distribucion de clases =================================================

def show_class_distribution(split_dir, names, title=''):
    counts = {name: 0 for name in names}
    for lbl in sorted((split_dir / 'train' / 'labels').glob('*.txt')):
        _, cls_ids = read_yolo_labels(lbl)
        for c in cls_ids:
            if 0 <= c < len(names):
                counts[names[c]] += 1
    total = sum(counts.values())
    print('  Total instancias en train: {}'.format(total))
    keys   = list(counts.keys())
    vals   = list(counts.values())
    colors = ['#d62728' if v < 50 else '#1f77b4' for v in vals]
    fig, ax = plt.subplots(figsize=(9, max(4, len(names) * 0.40)))
    bars = ax.barh(keys, vals, color=colors)
    ax.bar_label(bars, padding=3, fontsize=8)
    ax.set_xlabel('Instancias')
    ttl = 'Distribucion de clases (train)  -  rojo = <50 instancias'
    ax.set_title((title + '  |  ' + ttl) if title else ttl)
    plt.tight_layout()
    plt.show()
    low = [k for k, v in counts.items() if v < 50]
    if low:
        print('  Clases con <50 instancias: {}'.format(low))
    return counts

# === Visualizar augmentaciones ==============================================

def visualize_augmentations(split_dir, transform, names, n=16):
    imgs = sorted((split_dir / 'train' / 'images').glob('*'))
    rng  = random.Random(SEED)
    rng.shuffle(imgs)
    imgs   = imgs[:n]
    cols   = 4
    rows   = math.ceil(len(imgs) / cols)
    colors = _class_colors(len(names))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4), squeeze=False)
    axes_flat = axes.flatten()
    for ax, img_path in zip(axes_flat, imgs):
        lbl_path = split_dir / 'train' / 'labels' / (img_path.stem + '.txt')
        bboxes, cls_ids = read_yolo_labels(lbl_path)
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        if bboxes:
            aug     = transform(image=img, bboxes=bboxes, class_labels=cls_ids)
            img     = aug['image']
            bboxes  = list(aug['bboxes'])
            cls_ids = [int(c) for c in aug['class_labels']]  # Albumentations puede devolver floats
        h, w = img.shape[:2]
        for (cx, cy, bw, bh), cls in zip(bboxes, cls_ids):
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            color = colors[cls % len(colors)]
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            label = names[cls] if cls < len(names) else str(cls)
            cv2.putText(img, label, (x1, max(y1 - 4, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        ax.imshow(img)
        ax.axis('off')
    for ax in axes_flat[len(imgs):]:
        ax.axis('off')
    plt.suptitle('Preview augmentaciones - train split', fontsize=13)
    plt.tight_layout()
    plt.show()

# === Metricas RF-DETR =======================================================

def print_rfdetr_metrics(metrics, label=''):
    """Imprime metricas de RF-DETR en cualquier formato que devuelva la version instalada."""
    prefix = label + ' | ' if label else ''
    if metrics is None:
        print('  {}Metricas no disponibles.'.format(prefix))
        return
    if isinstance(metrics, dict):
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                print('  {}{:<20}: {:.4f}'.format(prefix, k, float(v)))
    else:
        for attr in ['mAP', 'map', 'mAP50', 'map50', 'mAP_50', 'AP50']:
            val = getattr(metrics, attr, None)
            if val is not None:
                print('  {}{}: {:.4f}'.format(prefix, attr, float(val)))
        print('  {}Objeto completo: {}'.format(prefix, metrics))

# === Guardar corrida RF-DETR en Drive =======================================

def save_rfdetr_run_to_drive(run_dir, drive_output_dir, model_name, metrics=None):
    run_dir = Path(run_dir)
    out_dir = Path(drive_output_dir) / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt = None
    for pattern in ['best*.pth', 'best*.pt', '*.pth', '*.pt']:
        candidates = sorted(run_dir.rglob(pattern))
        if candidates:
            best_candidates = [c for c in candidates if 'best' in c.name.lower()]
            best_ckpt = best_candidates[0] if best_candidates else candidates[-1]
            break
    if best_ckpt:
        shutil.copy2(best_ckpt, out_dir / 'best.pt')
        print('  Guardado: {} -> {}'.format(best_ckpt.name, out_dir / 'best.pt'))
    else:
        print('  Advertencia: no se encontro checkpoint en {}'.format(run_dir))
    for onnx_file in run_dir.rglob('*.onnx'):
        shutil.copy2(onnx_file, out_dir / 'best.onnx')
        print('  Guardado: {}'.format(out_dir / 'best.onnx'))
        break
    meta = {'model': model_name, 'date': datetime.now().isoformat()}
    if metrics is not None:
        if isinstance(metrics, dict):
            meta.update({k: float(v) for k, v in metrics.items() if isinstance(v, (int, float))})
        else:
            meta['metrics_repr'] = str(metrics)
    with open(out_dir / 'metrics.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print('  Guardado: metrics.json -> {}'.format(out_dir))

print('Utilidades cargadas')

## 4) Split de dataset

In [ ]:
if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
BASE_DIR.mkdir(parents=True, exist_ok=True)

names = load_names_from_yaml(DATASET + '/data.yaml')
print('Clases ({}): {}'.format(len(names), names))

stats = split_single_dataset(DATASET, SPLIT_DIR, VAL_RATIO, TEST_RATIO, SEED)
print('Split:', stats)

## 5) Distribución de clases

Revisa que todas las clases tengan suficientes instancias.
Las barras en **rojo** indican clases con <50 instancias.

In [ ]:
print('=== Distribucion de clases (Cards dataset) ===')
show_class_distribution(SPLIT_DIR, names, 'RF-DETR')

## 6) Preview de augmentaciones

Visualiza cómo quedarán las imágenes augmentadas **antes** de aplicarlas.
Verifica que los bounding boxes sigan siendo correctos.

In [ ]:
augment_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.35),
    A.HueSaturationValue(hue_shift_limit=3, sat_shift_limit=10, val_shift_limit=15, p=0.25),
    A.GaussNoise(p=0.20),
    A.MotionBlur(blur_limit=3, p=0.15),
    A.Affine(scale=(0.90, 1.10), translate_percent=(0.0, 0.04),
             rotate=(-8, 8), shear=(-3, 3), p=0.40),
    A.RandomShadow(p=0.15),
    A.CLAHE(p=0.15),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(1, 32), hole_width_range=(1, 32), p=0.20),
    A.ImageCompression(quality_range=(60, 100), p=0.15),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.2))

visualize_augmentations(SPLIT_DIR, augment_transform, names, n=16)

## 7) Aplicar augmentación offline

In [ ]:
if USE_OFFLINE_ALBUMENTATIONS:
    print('Aplicando Albumentations...')
    created = apply_offline_augment(SPLIT_DIR, AUG_DIR, AUG_MULTIPLIER, augment_transform)
    DATA_ROOT = AUG_DIR
    print('Imagenes nuevas creadas: {}'.format(created))
else:
    DATA_ROOT = SPLIT_DIR
    print('Albumentations offline desactivado.')

counts = {s: len(list((DATA_ROOT / s / 'images').glob('*'))) for s in ['train', 'val', 'test']}
print('train:{} | val:{} | test:{}'.format(counts['train'], counts['val'], counts['test']))

## 8) Crear data.yaml

RF-DETR acepta formato YOLO de forma nativa. Se crea el `data.yaml` estándar y, por compatibilidad,
se crea un symlink `valid/` apuntando a `val/` por si la versión instalada lo requiere.

In [ ]:
# RF-DETR detecta el formato del dataset buscando 'data.yaml' o 'data.yml'
# DENTRO del directorio que se le pasa como dataset_dir.
# Por eso el yaml debe crearse en DATA_ROOT, no en BASE_DIR.
DATA_YAML = DATA_ROOT / 'data.yaml'
build_yaml(DATA_ROOT, names, DATA_YAML)
print('YAML creado en:', DATA_YAML)
print(open(DATA_YAML).read())

# Compatibilidad: algunos builds de RF-DETR buscan 'valid/' en lugar de 'val/'
valid_dir = DATA_ROOT / 'valid'
if not valid_dir.exists():
    shutil.copytree(DATA_ROOT / 'val', valid_dir)
    print('Creado valid/ como copia de val/ (compatibilidad RF-DETR)')

RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset listo en:', DATA_ROOT)

## 9) Sección A — Entrenar RF-DETR Nano

`RFDETRNano` es el modelo más pequeño. Cabe cómodamente en la T4 de Colab con `batch_size=4`.

In [ ]:
model_nano = RFDETRNano()
model_nano.train(
    dataset_dir=str(DATA_ROOT),
    epochs=EPOCHS,
    batch_size=BATCH,
    grad_accum_steps=GRAD_ACCUM,
    lr=LR,
    resolution=IMG_SIZE_NANO,
    output_dir=str(RUNS_DIR / 'rfdetr_nano'),
)

## 10) Evaluar y exportar RF-DETR Nano

In [ ]:
import pandas as pd

def read_rfdetr_metrics(run_dir):
    run_dir = Path(run_dir)
    candidates = [
        run_dir / 'metrics.csv',
        run_dir / 'lightning_logs' / 'version_0' / 'metrics.csv',
        *sorted(run_dir.rglob('metrics.csv')),
    ]
    for path in candidates:
        if path.exists():
            return path
    return None

metrics_nano = None
csv_path = read_rfdetr_metrics(RUNS_DIR / 'rfdetr_nano')
if csv_path:
    df = pd.read_csv(csv_path)
    val_rows = df[df['val/mAP_50'].notna()] if 'val/mAP_50' in df.columns else pd.DataFrame()
    if not val_rows.empty:
        best_row  = val_rows.loc[val_rows['val/mAP_50'].idxmax()]
        mAP50     = float(best_row['val/mAP_50'])
        mAP50_95  = float(best_row['val/mAP_50_95']) if 'val/mAP_50_95' in df.columns else float('nan')
        print('RF-DETR Nano — mejor epoca {}:'.format(int(best_row['epoch'])))
        print('  mAP50:    {:.4f}'.format(mAP50))
        print('  mAP50-95: {:.4f}'.format(mAP50_95))
        metrics_nano = {'mAP50': mAP50, 'mAP50_95': mAP50_95}
    else:
        print('No se encontraron filas de validacion en el CSV.')
else:
    print('metrics.csv no encontrado.')
    print('Archivos en run dir:')
    for f in sorted((RUNS_DIR / 'rfdetr_nano').rglob('*')):
        print(' ', f.relative_to(RUNS_DIR / 'rfdetr_nano'))

onnx_path = RUNS_DIR / 'rfdetr_nano' / 'inference_model.onnx'
if not onnx_path.exists():
    try:
        model_nano.export(output_dir=str(RUNS_DIR / 'rfdetr_nano'))
        print('Export ONNX completado.')
    except Exception as e:
        print('Export no disponible: {}'.format(e))
else:
    print('ONNX ya existe:', onnx_path)

save_rfdetr_run_to_drive(RUNS_DIR / 'rfdetr_nano', DRIVE_OUTPUT_DIR, 'rfdetr_nano', metrics_nano)

## 11) Sección B — Entrenar RF-DETR Small

`RFDETRSmall` es más preciso que Nano pero usa más VRAM. En T4 de Colab, `batch_size=4` suele ser seguro.

In [ ]:
model_small = RFDETRSmall()
model_small.train(
    dataset_dir=str(DATA_ROOT),
    epochs=EPOCHS,
    batch_size=BATCH,
    grad_accum_steps=GRAD_ACCUM,
    lr=LR,
    resolution=IMG_SIZE_SMALL,
    output_dir=str(RUNS_DIR / 'rfdetr_small'),
)

## 12) Evaluar y exportar RF-DETR Small

In [ ]:
metrics_small = None
csv_path = read_rfdetr_metrics(RUNS_DIR / 'rfdetr_small')
if csv_path:
    df = pd.read_csv(csv_path)
    val_rows = df[df['val/mAP_50'].notna()] if 'val/mAP_50' in df.columns else pd.DataFrame()
    if not val_rows.empty:
        best_row  = val_rows.loc[val_rows['val/mAP_50'].idxmax()]
        mAP50     = float(best_row['val/mAP_50'])
        mAP50_95  = float(best_row['val/mAP_50_95']) if 'val/mAP_50_95' in df.columns else float('nan')
        print('RF-DETR Small — mejor epoca {}:'.format(int(best_row['epoch'])))
        print('  mAP50:    {:.4f}'.format(mAP50))
        print('  mAP50-95: {:.4f}'.format(mAP50_95))
        metrics_small = {'mAP50': mAP50, 'mAP50_95': mAP50_95}
    else:
        print('No se encontraron filas de validacion en el CSV.')
else:
    print('metrics.csv no encontrado.')
    print('Archivos en run dir:')
    for f in sorted((RUNS_DIR / 'rfdetr_small').rglob('*')):
        print(' ', f.relative_to(RUNS_DIR / 'rfdetr_small'))

onnx_path = RUNS_DIR / 'rfdetr_small' / 'inference_model.onnx'
if not onnx_path.exists():
    try:
        model_small.export(output_dir=str(RUNS_DIR / 'rfdetr_small'))
        print('Export ONNX completado.')
    except Exception as e:
        print('Export no disponible: {}'.format(e))
else:
    print('ONNX ya existe:', onnx_path)

save_rfdetr_run_to_drive(RUNS_DIR / 'rfdetr_small', DRIVE_OUTPUT_DIR, 'rfdetr_small', metrics_small)

## 13) Comparación de modelos RF-DETR

Ejecutar sólo después de ambas secciones de entrenamiento.

In [ ]:
print('{:<18} {:>10} {:>10} {:>12}'.format('Modelo', 'mAP50', 'mAP50-95', 'Pesos'))
print('-' * 54)

model_runs = [
    ('RF-DETR Nano',  'metrics_nano',  'rfdetr_nano'),
    ('RF-DETR Small', 'metrics_small', 'rfdetr_small'),
]

g = globals()
for mname, var, run_name in model_runs:
    m = g.get(var)
    if m is None:
        # Leer desde metrics.json en Drive
        json_path = Path(DRIVE_OUTPUT_DIR) / run_name / 'metrics.json'
        if json_path.exists():
            with open(json_path) as f:
                meta = json.load(f)
            m = meta
        else:
            m = {}

    map50   = m.get('mAP50', m.get('mAP', m.get('map', float('nan'))))
    map5095 = m.get('mAP50_95', m.get('mAP50-95', float('nan')))

    pts = sorted((RUNS_DIR / run_name).rglob('*.pt')) + sorted((RUNS_DIR / run_name).rglob('*.pth'))
    mb  = os.path.getsize(str(pts[-1])) / 1e6 if pts else float('nan')

    map50_s   = '{:.4f}'.format(map50)   if map50   == map50   else 'N/A'
    map5095_s = '{:.4f}'.format(map5095) if map5095 == map5095 else 'N/A'
    print('{:<18} {:>10} {:>10} {:>9.1f} MB'.format(mname, map50_s, map5095_s, mb))

## Recomendaciones

### RF-DETR vs YOLO — cuándo usar cada uno
| | RF-DETR | YOLO |
|---|---|---|
| Objetos superpuestos | Mejor | Bueno |
| Inferencia en tiempo real | Más lento | Más rápido |
| VRAM requerida | Más alta | Más baja |
| Entrenamiento | Más rápido (menos epochs) | Más lento |

### Ajustes comunes
- **OOM (Out of Memory)**: reduce `BATCH` a `2` y sube `GRAD_ACCUM` a `8` para mantener batch efectivo = 16.
- **Resolución**: cada checkpoint tiene sus propios positional embeddings y **no se puede cambiar** la resolución sin reentrenar desde cero.
  - `rf-detr-nano.pth` → `IMG_SIZE_NANO = 384` (24×24 patches)
  - `rf-detr-small.pth` → `IMG_SIZE_SMALL = 512` (32×32 patches)
  Cualquier otro valor produce `RuntimeError: size mismatch for position_embeddings`.
- **Export ONNX manual**: si `model.export()` no está disponible en tu versión,
  usa `torch.onnx.export(model_nano.model, dummy_input, 'model.onnx')`.
- **Drive**: `best.pt` y `best.onnx` se guardan en `Cards_Training_Output/rfdetr_nano/` y `rfdetr_small/`.